# Data Cleaning, Standardization & Integration Documentation

## Customer Spending Behavior & Revenue Analytics

### Purpose

This notebook documents the data cleaning, standardization, transformation, and multi source integration process used to prepare the final datasets for analysis.

The preparation process includes:

- data quality assessment
- data type standardization
- cancellation handling
- invalid price handling
- missing-value handling
- duplicate removal
- revenue calculation
- product category engineering
- customer level dataset preparation
- integration of retail data with World Bank population data
- validation of the integrated dataset

The main cleaning and feature engineering work is implemented in:

`notebooks/01_data_understanding_cleaning_feature_engineering.ipynb`

Multi source integration is implemented in:

`scripts/merge_retail_world_bank.py`

# Part A — Retail Data Cleaning and Preparation

## 1. Data Quality Assessment

Before cleaning the dataset, several data-quality checks were performed.

The raw combined dataset contained **1,067,371 rows** covering the period from December 2009 to December 2011.

The assessment included:

- checking exact duplicate rows
- measuring missing values and percentages
- examining Quantity and Price distributions
- checking the date range
- counting unique customers, products, invoices and countries
- identifying cancelled transactions
- identifying zero and negative prices
- identifying negative quantities
- investigating extreme Quantity and Price values

The investigation showed that different data quality issues required different treatments rather than applying a single rule to all records.

## 2. Data Type and Format Standardization

Data types and formats were standardized before analysis.

The following transformations were applied:

- `InvoiceDate` was converted to datetime
- `StockCode` was converted to string
- `StockCode` was standardized to uppercase
- `Invoice` was converted to string

These transformations ensure consistent formats for filtering, joining, date analysis and product identification.

## 3. Missing Customer IDs

Approximately 22.7% of the raw rows had a missing `Customer ID`.

These transactions were not automatically removed from the complete sales dataset because they can still contribute to overall revenue analysis.

For customer level analysis, a separate dataset was created containing only records with a valid Customer ID.

This preserves useful transaction information while ensuring that customer level metrics are calculated only for identifiable customers.

## 4. Cancellation Handling

Cancelled transactions were identified using invoice numbers beginning with `C`.

A Boolean field called `is_cancelled` was created and the data was separated into:

- completed sales
- cancelled transactions

Cancelled transactions were preserved in a separate dataset rather than deleted because cancellation behavior is itself an important business metric.

This allows completed revenue analysis and cancellation analysis to be performed separately.

## 5. Invalid Values and Duplicate Handling

For completed sales, rows with a zero or negative `Price` were removed because they do not represent valid positive revenue sales for the main revenue analysis.

Missing product descriptions were filled with:

`Unknown Product`

Exact duplicate rows were identified and removed using `drop_duplicates()`.

After separating cancellations, an additional validation check confirmed that negative quantities had not remained in the completed-sales dataset.

## 6. Revenue Calculation

A new analytical feature called `Revenue` was created:

`Revenue = Quantity × Price`

This converts transaction level quantity and unit price information into a monetary measure that can be aggregated by customer, country, product category and time period.

## 7. Product Category Engineering

The raw dataset did not contain a product category field.

A `Category` feature was therefore engineered from product descriptions using keyword based classification.

The classification rules were refined iteratively by reviewing both:

- frequently occurring uncategorized products
- high revenue uncategorized products

The final classification contains 14 categories.

The remaining `Other` category represented 9.55% of transactions and 5.86% of revenue, representing a relatively small long tail of miscellaneous products.

# Part B — Multi Source Data Integration

## 8. Multi Source Data Integration

The cleaned retail data was enriched with population data retrieved from the World Bank REST API.

The retail dataset and World Bank dataset used different country naming conventions.

Country names were therefore standardized before integration. Examples include:

- `EIRE` → `Ireland`
- `RSA` → `South Africa`
- `USA` → `United States`
- `Korea` → `Korea, Rep.`
- `Hong Kong` → `Hong Kong SAR, China`
- `Czech Republic` → `Czechia`

The retail and population datasets were then joined using country and year.

## 9. Join Strategy

A `LEFT JOIN` was used to integrate the World Bank population data with the retail dataset.

The retail dataset was used as the left dataset because all retail transactions needed to be preserved even when no population match was available.

Records without a reliable population match were retained with a missing population value rather than being deleted or assigned an incorrect value.

## 10. Integration Validation

The integrated dataset was validated after the merge.

Validation results:

- Original retail rows: **1,067,371**
- Integrated rows: **1,067,371**
- Population matched: **1,064,836**
- Population unmatched: **2,535**
- Population match rate: **99.76%**

All retail transaction rows were therefore preserved.

The remaining unmatched population records correspond to geographic categories that could not be reliably mapped to individual World Bank countries, including:

- Channel Islands
- Unspecified
- European Community
- West Indies

These values were deliberately left unmatched rather than assigning potentially incorrect population data.

## 11. Outputs and Reproducibility

The cleaning and integration process produces reusable datasets in `data/processed/`, including:

- `online_retail_cleaned.csv`
- `cancellations.csv`
- `online_retail_with_customer.csv`
- `customers.csv`
- `invoices.csv`
- `invoice_items.csv`
- `products.csv`
- `retail_with_population.csv`

The extraction and integration scripts are stored in `scripts/`.

The project notebooks, SQL files and Python scripts are version controlled with Git, allowing the data preparation workflow to be documented and reproduced.

## 12. Conclusion

The data-preparation workflow transforms raw transaction data into clean and analysis ready datasets through:

- data quality assessment
- data type and format standardization
- missing value handling
- cancellation separation
- invalid value handling
- duplicate removal
- feature engineering
- multi source aggregation
- country name standardization
- join validation

The integration process preserves all **1,067,371** retail transactions while successfully matching **99.76%** of records with World Bank population data.

The workflow is documented and version controlled with Git.